# **YouTube Topic Researcher (simple version)**

Your podcast summarizer, extended in two steps:

1. **Summarize each video** about a topic. Videos can be in any language, and each summary says how strong the evidence is.
2. **Compare all the summaries.** Where do the videos agree, where do they disagree, and which claims have no real evidence?

You give it a topic. It finds the videos itself, or you paste your own links.

In [ ]:
!pip install -q youtube-transcript-api yt-dlp openai

## Setup
Uses the same `OPENAI_TOKEN` Colab secret as your other notebooks.

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from openai import OpenAI
from google.colab import userdata
from IPython.display import Markdown, display
import re

openai_client = OpenAI(api_key=userdata.get('OPENAI_TOKEN'))
MODEL = "gpt-4o-mini"
ytt = YouTubeTranscriptApi()

## Step 1: Find videos and get transcripts (any language)

In [ ]:
def search_youtube(query, how_many=5):
    """Search YouTube without an API key. Returns a list of (title, url)."""
    import yt_dlp
    with yt_dlp.YoutubeDL({"quiet": True, "extract_flat": True}) as ydl:
        results = ydl.extract_info(f"ytsearch{how_many}:{query}", download=False)
    return [(v["title"], f"https://www.youtube.com/watch?v={v['id']}") for v in results["entries"]]


def get_video_id(url):
    match = re.search(r"(?:v=|youtu\.be/|shorts/)([0-9A-Za-z_-]{11})", url)
    if match:
        return match.group(1)
    raise ValueError("Invalid YouTube URL")


def get_transcript(url):
    """Gets the transcript in whatever language the video has (English preferred)."""
    video_id = get_video_id(url)
    transcripts = list(ytt.list(video_id))
    english = [t for t in transcripts if t.language_code.startswith("en")]
    chosen = (english or transcripts)[0]
    text = " ".join(snippet.text for snippet in chosen.fetch())
    return text[:200_000]  # very long videos get cut off so they fit in the model's context

## Step 2: Summarize one video, with an evidence rating for each claim

In [ ]:
video_prompt = """You analyze YouTube video transcripts (they can be in ANY language) about a given TOPIC.
Always answer in English, in Markdown.

List the main claims about the TOPIC. For each claim give:
- **Claim:** 1 sentence
- **Evidence:** what the speaker cites (study, statistic, expert, personal story) or "none"
- **Strength:** Strong (research/data) / Medium (expert reasoning) / Weak (anecdote or no evidence)

Then add:
- **Sponsors / products sold:** anything the speaker promotes, or "none"
- **Red flags:** exaggerations, miracle claims, fear-selling, or "none"

Ignore ads, small talk and anything off-topic. If the video is not about the TOPIC, just say "NOT RELEVANT"."""


def summarize_video(topic, title, url):
    transcript = get_transcript(url)
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": video_prompt},
            {"role": "user", "content": f"TOPIC: {topic}\nVIDEO: {title}\n\nTranscript:\n{transcript}"},
        ],
    )
    return response.choices[0].message.content

## Step 3: Compare all the videos

In [ ]:
compare_prompt = """You get summaries of several YouTube videos about one TOPIC.
Write a short Markdown report for someone trying to make a real decision:

## Bottom line
3-4 sentences in plain language. Be honest about uncertainty.
## What most videos agree on
Note whether the agreement is backed by research or only by opinion.
## Where they disagree
## Claims with weak or no evidence
## Who is selling something
## Practical takeaways
For health, money or legal topics, say what to check with a professional.

Refer to videos by their number, e.g. (Video 2). Do not invent facts that are not in the summaries."""


def research_topic(topic, urls=None, how_many=5):
    # 1. Find videos (or use the links you gave)
    if urls:
        videos = [(url, url) for url in urls]
    else:
        videos = search_youtube(topic, how_many)

    # 2. Summarize each one
    summaries = []
    for i, (title, url) in enumerate(videos, 1):
        try:
            summary = summarize_video(topic, title, url)
        except Exception as e:
            print(f"Skipped {title}: no transcript ({type(e).__name__})")
            continue
        if "NOT RELEVANT" in summary[:50]:
            print(f"Skipped {title}: not about the topic")
            continue
        print(f"Done: {title}")
        summaries.append(f"### Video {i}: {title}\n{url}\n\n{summary}")

    if not summaries:
        print("No usable videos found. Try another topic or paste links with urls=[...]")
        return

    # 3. Compare them
    all_summaries = "\n\n---\n\n".join(summaries)
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": compare_prompt},
            {"role": "user", "content": f"TOPIC: {topic}\n\n{all_summaries}"},
        ],
    )
    report = response.choices[0].message.content

    display(Markdown(f"# {topic}\n\n{report}\n\n---\n# Individual videos\n\n{all_summaries}"))

## Try it
Option A: give it a topic and it finds the videos. To include other countries, write the topic in another language, for example `"Kreatin Gehirn"` (German) or `"creatina cerebro"` (Spanish).

In [ ]:
research_topic("creatine for brain health", how_many=4)

Option B: paste your own videos.

In [ ]:
research_topic("creatine for brain health", urls=[
    "https://www.youtube.com/watch?v=JCTb3QSrGMQ",
])